# Data exploration and analysis

Ordered as: corpus overview -> classification-input pipeline -> labeling status -> interrater agreement analysis. Companion to `measurement/impoliteness_pilot.ipynb` (the actual scoring pipeline) and `measurement/compute_interrater_agreement.py` (the agreement CLI) -- this notebook is for exploring and reading the results, not reproducing the scoring.

In [2]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/My Drive/my_projects/M.A. Parliament/Code and Data/data"
else:
    if "DATA_ROOT" not in os.environ:
        try:
            from dotenv import load_dotenv, find_dotenv
            load_dotenv(find_dotenv())
        except ImportError:
            pass
    DATA_ROOT = os.environ.get("DATA_ROOT", "")

assert DATA_ROOT and os.path.isdir(DATA_ROOT), f"DATA_ROOT not set or missing: {DATA_ROOT!r}"

In [3]:
import pandas as pd
from pathlib import Path

_cwd = Path(".").resolve()
_repo = _cwd if (_cwd / "preprocessing" / "nsc_rule_parser.py").exists() else _cwd.parent
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

## Part 1 — Corpus overview

Full `stateparl_v3_paragraphs.parquet` (all 16 states, all affiliations, 2000-2025).

In [4]:
V3 = Path(DATA_ROOT) / "raw" / "stateparl_v3_parquet"
paragraphs = pd.read_parquet(V3 / "stateparl_v3_paragraphs.parquet")
paragraphs["date"] = pd.to_datetime(paragraphs["date"])
print(f"{len(paragraphs):,} paragraphs, {paragraphs['state'].nunique()} states, "
      f"{paragraphs['date'].min().date()} to {paragraphs['date'].max().date()}")

16,078,467 paragraphs, 16 states, 2000-01-19 to 2025-12-18


### Cadence: how many paragraphs per session / week / month / year / period

"Session" = one protocol (`protocol_id`); "period" = one state's legislative term (`state`+`period`, since period numbers repeat across states). Week/month/year averages are over units that actually had >=1 sitting (recess gaps aren't zero-filled).

In [5]:
cadence = {
    "session": paragraphs.groupby("protocol_id").size().mean(),
    "week":    paragraphs.groupby(paragraphs["date"].dt.to_period("W")).size().mean(),
    "month":   paragraphs.groupby(paragraphs["date"].dt.to_period("M")).size().mean(),
    "year":    paragraphs.groupby(paragraphs["date"].dt.year).size().mean(),
    "period":  paragraphs.groupby(["state", "period"]).size().mean(),
}
for unit, avg in cadence.items():
    print(f"avg paragraphs per {unit:8s}: {avg:,.0f}")

avg paragraphs per session : 1,694
avg paragraphs per week    : 13,993
avg paragraphs per month   : 51,534
avg paragraphs per year    : 618,403
avg paragraphs per period  : 162,409


## Part 2 — Building the classification input

Same pool `impoliteness_pilot.ipynb` scores from -- built once in `impoliteness_lib.build_classification_pool()`, not duplicated here. See that function's docstring for the full rationale behind each step.

In [6]:
sys.path.insert(0, str(_repo / "measurement"))
from impoliteness_lib import build_classification_pool

pool = build_classification_pool(DATA_ROOT)
_, _, _, _, merged, dedup = pool

paragraphs: 16,078,467 rows, 16 states
nsc.parquet: 4,920,872 segments -> 4,825,846 after excluding non-interjection row types
para_window (+/-1yr around each state's AfD entry): 1,223,597 of 16,078,467 paragraphs
merged: 1,277,020 rows; classifiable+deduplicated: 1,144,158


### Paragraph-length stats (context-window sizing)

In [7]:
lens = dedup["text_to_classify"].str.len()
print(lens.describe(percentiles=[.5, .9, .99, .999]))
print(f"max chars: {lens.max():,}  (~{lens.max() // 4:,} tokens, chars/4 estimate)")

count    1.144158e+06
mean     2.222605e+02
std      2.320913e+02
min      1.000000e+00
50%      1.500000e+02
90%      5.360000e+02
99%      9.710000e+02
99.9%    1.551843e+03
max      5.383000e+03
Name: text_to_classify, dtype: float64
max chars: 5,383  (~1,345 tokens, chars/4 estimate)


## Part 3 — Labeling status

Every `impoliteness_pilot_predictions_*.csv` produced so far (one row per model x paragraph).

In [8]:
import glob

pred_files = sorted(glob.glob(str(Path(DATA_ROOT) / "measurement" / "impoliteness_pilot_predictions_*.csv")))
preds = [pd.read_csv(p) for p in pred_files]
for p, df in zip(pred_files, preds):
    print(f"{df['model_name'].iloc[0]:25s} n={len(df):>5}  {Path(p).name}")

all_labeled = pd.concat(preds, ignore_index=True)
print(f"\nTotal label events (rows): {len(all_labeled):,}")
print(f"Unique paragraphs labeled by >=1 model: {all_labeled['paragraph_id'].nunique():,}")

gemma4:12b                n= 2000  impoliteness_pilot_predictions_2026-07-28_gemma4-12b_n2000.csv
llama3.1:8b               n= 2000  impoliteness_pilot_predictions_2026-07-28_llama3-1-8b_n2000.csv
mistral-small3.2:24b      n= 2000  impoliteness_pilot_predictions_2026-07-28_mistral-small3-2-24b_n2000.csv
qwen3:14b-q4_K_M          n= 2000  impoliteness_pilot_predictions_2026-07-28_qwen3-14b-q4_K_M_n2000.csv
qwen3:14b-q4_K_M          n=  500  impoliteness_pilot_predictions_2026-07-28_qwen3-14b-q4_K_M_n500.csv

Total label events (rows): 8,500
Unique paragraphs labeled by >=1 model: 2,000


## Part 4 — Interrater agreement (n=2000 run, 4 models)

Reuses `cohen_kappa` from `measurement/compute_interrater_agreement.py` rather than reimplementing it; Krippendorff's alpha via the `krippendorff` package (see that script for why: hand-rolling the coincidence-matrix math isn't worth the risk for a number that goes in the thesis).

In [12]:
from compute_interrater_agreement import cohen_kappa
import krippendorff

n2000_files = [p for p in pred_files if p.endswith("_n2000.csv")]
wide = None
for p in n2000_files:
    df = pd.read_csv(p)
    model = df["model_name"].iloc[0]
    col = df[["paragraph_id", "impolite", "content", "reason"]].rename(
        columns={"impolite": model, "content": "content_" + model, "reason": "reason_" + model})
    wide = col if wide is None else wide.merge(col.drop(columns=["content_" + model]), on="paragraph_id", how="outer")
models = [df["model_name"].iloc[0] for df in (pd.read_csv(p) for p in n2000_files)]

print("Impoliteness rate per model:")
for m in models:
    print(f"  {m:25s} {wide[m].mean():.1%}")

Impoliteness rate per model:
  gemma4:12b                7.4%
  llama3.1:8b               5.7%
  mistral-small3.2:24b      5.5%
  qwen3:14b-q4_K_M          7.0%


In [10]:
print("Pairwise Cohen's kappa:")
complete = wide.dropna(subset=models)
for i, m1 in enumerate(models):
    for m2 in models[i + 1:]:
        k = cohen_kappa(complete[m1].astype(bool).to_numpy(), complete[m2].astype(bool).to_numpy())
        print(f"  {m1} vs {m2}: {k:.3f}")

alpha = krippendorff.alpha(
    reliability_data=wide[models].to_numpy(dtype=float).T, level_of_measurement="nominal")
print(f"\nKrippendorff's alpha (joint, n={len(wide):,}): {alpha:.3f}")

Pairwise Cohen's kappa:
  gemma4:12b vs llama3.1:8b: 0.464
  gemma4:12b vs mistral-small3.2:24b: 0.553
  gemma4:12b vs qwen3:14b-q4_K_M: 0.571
  llama3.1:8b vs mistral-small3.2:24b: 0.511
  llama3.1:8b vs qwen3:14b-q4_K_M: 0.488
  mistral-small3.2:24b vs qwen3:14b-q4_K_M: 0.554

Krippendorff's alpha (joint, n=2,000): 0.524


### A few actual disagreement cases

Paragraphs where the 4 models didn't unanimously agree -- read these directly rather than trusting a summary statistic.

In [14]:
disagreement = wide[wide[models].nunique(axis=1) > 1]
print(f"{len(disagreement):,} of {len(wide):,} paragraphs have >=1 model disagreeing\n")

for _, row in disagreement.sample(n=100, random_state=1).iterrows():
    print(row["content_" + models[0]][:200])
    for m in models:
        print(f"  {m:25s} impolite={row[m]!s:5}  reason: {row['reason_' + m]}")
    print()

208 of 2,000 paragraphs have >=1 model disagreeing

Frage 2: Homo- und Transphobie. Gut, nehmen wir einmal an, es gäbe diese Geisteskrankheit. Dann wären doch diejenigen, die unter dieser Krankheit litten, in ihrem freien Willen beeinträchtigt. Sie wär
  gemma4:12b                impolite=True   reason: Die Verwendung des Begriffs 'Geisteskrankheit' als rhetorisches Mittel zur Abwertung einer Gruppe sowie der sarkastische Unterton stellen einen unhöflichen Stil dar.
  llama3.1:8b               impolite=False  reason: Die Frage stellt eine sachliche und neutrale Überlegung zu einer hypothetischen medizinischen Diagnose an.
  mistral-small3.2:24b      impolite=False  reason: Die Äußerung enthält diskriminierende Inhalte, aber keine stilistischen Marker der Unhöflichkeit.
  qwen3:14b-q4_K_M          impolite=False  reason: Die Äußerung ist sachlich formuliert und enthält keine unhöflichen Marker.

Herr Staatssekretär Weinberg hat gesagt, ich soll nicht sagen, dass Sie mit der Altersversor

Llama -> unterinkludiert
Gemma -> überinkludiert, aber gut mit irony etc.
Qwen: gut mit sarkasmus, aber überinkludiert
mistral: erkennt policy kritik, erkennt nicht angriffe, unterinkludiert.

EDGE CASES DEFINIEREN

In the end ensemle decisions.


Can I train on reasonings? I can generate both with GPT 4.0, "gold" labels and "gold reasons".

These models are mulitmodal. In the future I really want to analyze videos.

The "Unified" in Gemma 4 12B Unified refers to its encoder-free architecture. Other Gemma 4 models use dedicated encoders to process multimodal data before passing it to the LLM. https://huggingface.co/google/gemma-4-12B-it


## Part 5 — AfD entry: co-occurring party changes (confound check for DiD/FE)

`DATA_ROOT/raw/afd_entry.xlsx` (the curated treatment variable) records only AfD's own entry/exit
year per state, not what else changed in the chamber at that moment. This checks, per state,
whether any other PPG (party group; codebook Table A.6, excludes functional codes
`pre`/`gov`/`oth`/`nsc`/`ind`) newly appears or disappears in the *same* period transition as AfD
-- a DiD/FE design that treats "AfD entry" as an isolated shock will misattribute any rhetoric
shift that's really coming from a changed party mix.

In [ ]:
FUNCTIONAL = {"pre", "gov", "oth", "nsc", "ind"}
ppg = paragraphs[~paragraphs["affiliation"].isin(FUNCTIONAL)].dropna(subset=["affiliation"])

periods = (
    ppg.groupby(["state", "period"])
    .agg(parties=("affiliation", lambda s: frozenset(s.unique())), start=("date", "min"))
    .reset_index()
    .sort_values(["state", "period"])
)

VERB = {"entry": "ENTERS", "exit": "EXITS"}

def afd_transitions(sub, direction):
    prev = None
    for _, row in sub.iterrows():
        if prev is not None:
            had, has = "afd" in prev, "afd" in row["parties"]
            fires = (not had and has) if direction == "entry" else (had and not has)
            if fires:
                entered = (row["parties"] - prev) - {"afd"}
                exited = (prev - row["parties"]) - {"afd"}
                print(f"{row['state']} period {row['period']} ({row['start'].date()}): AfD {VERB[direction]} -- "
                      f"also entering: {sorted(entered) or 'none'}; also exiting: {sorted(exited) or 'none'}")
        prev = row["parties"]

print("=== AfD entry transitions ===")
for state, sub in periods.groupby("state"):
    afd_transitions(sub.sort_values("period"), "entry")

print("\n=== AfD exit transitions ===")
for state, sub in periods.groupby("state"):
    afd_transitions(sub.sort_values("period"), "exit")

=== AfD entry transitions ===
bb period 6 (2014-10-08): AfD ENTERS -- also entering: ['frw']; also exiting: ['fdp']
be period 18 (2016-10-27): AfD ENTERS -- also entering: ['fdp']; also exiting: ['pir']
bw period 16 (2016-05-11): AfD ENTERS -- also entering: none; also exiting: none
by period 18 (2018-11-05): AfD ENTERS -- also entering: ['fdp']; also exiting: none
hb period 19 (2015-07-01): AfD ENTERS -- also entering: ['bbr', 'fdp', 'lkr']; also exiting: none
he period 20 (2019-01-18): AfD ENTERS -- also entering: none; also exiting: none
hh period 21 (2015-09-16): AfD ENTERS -- also entering: none; also exiting: none
mv period 7 (2016-10-04): AfD ENTERS -- also entering: ['bmv', 'frw']; also exiting: ['grn', 'npd']
ni period 18 (2017-11-14): AfD ENTERS -- also entering: none; also exiting: none
nw period 17 (2017-06-27): AfD ENTERS -- also entering: none; also exiting: ['pir']
rp period 17 (2016-05-18): AfD ENTERS -- also entering: ['fdp']; also exiting: none
sh period 19 (2017-06-0

Reading the entry transitions: 5 of 16 states are clean (BW '16, HE '19, HH '15, NI '17, ST '16 --
nothing else changes). In the rest, two recurring confounds dominate:

- **FDP** moves in lockstep with AfD's debut, but in opposite directions by region: it *exits*
  alongside AfD's entry in the three 2014 eastern cases (BB, SN, TH -- FDP falling below 5% while
  AfD clears it), but *re-enters* alongside AfD in four western cases (BE '16, BY '18, HB '15,
  RP '16). Same underlying national FDP trough-then-rebound, opposite compositional effect
  depending on the state.
- **Piratenpartei** exits in the same transition as AfD enters in 4 states (BE '16, NW '17, SH '17,
  SL '17) -- plausibly the same protest-vote channel migrating from one insurgent party to the
  other, not an independent confound, but still a simultaneous compositional shift to control for.

MV '16 and SL '17 also lose an established party (Grüne) at the same transition, and MV/SN lose
NPD (a direct predecessor on the far right) -- worth flagging separately since that's arguably part
of the AfD story rather than a nuisance confound. HB '15 is messier still: `lkr`/`bbr` entering
there reflects a mid-period AfD splinter, not a simultaneous election event, so don't read it as a
same-election co-entry.

On exits: SH's 2022 AfD exit is clean (nothing else changes). HB's 2023 AfD exit is dominated by
AfD's own internal breakaway groups (`biw`/`lfm`/`lkr`/`mrf` exiting, `bag`/`büd` entering) rather
than by other established parties -- not very informative as an exit-side counterfactual.